# pyflakes — unused imports / undefined names (raw CLI output)

Install: `pip install pyflakes` | CLI: `python -m pyflakes <file>...` (no built-in directory recursion; this notebook batches `*.py` paths).

Exit code **1** means pyflakes reported issues; **0** means clean. Raw messages go to **stdout**.

In [1]:
import pyflakes
print(dir(pyflakes))

['__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__path__', '__spec__', '__version__']


In [2]:
import importlib
import pkgutil

for finder, name, ispkg in pkgutil.walk_packages(pyflakes.__path__, pyflakes.__name__ + '.', onerror=lambda x: None):
    try:
        mod = importlib.import_module(name)
        print(name, '->', dir(mod))
    except BaseException as e:
        print(name, '-> SKIP:', e)

pyflakes.__main__ -> ['__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__spec__', 'main']
pyflakes.api -> ['PYTHON_SHEBANG_REGEX', '__all__', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__spec__', '__version__', '_exitOnSignal', '_get_version', 'ast', 'check', 'checkPath', 'checkRecursive', 'checker', 'isPythonFile', 'iterSourceCode', 'main', 'modReporter', 'os', 'platform', 're', 'sys']
pyflakes.checker -> ['Annotation', 'AnnotationState', 'Argument', 'Assignment', 'Binding', 'Builtin', 'CONVERSION_FLAG_RE', 'Checker', 'ClassDefinition', 'ClassScope', 'Definition', 'DetectClassScopedMagic', 'DoctestScope', 'ExportBinding', 'FOR_TYPES', 'FunctionDefinition', 'FunctionScope', 'FutureImportation', 'GeneratorScope', 'Importation', 'ImportationFrom', 'LENGTH_RE', 'MAPPING_KEY_RE', 'ModuleScope', 'NamedExprAssignment', 'PRECISION_RE', 'PYPY', 'Scope', 'StarImportation', 'SubmoduleImportation', '

In [3]:
# Raw CLI: version (stdout)
import subprocess
import sys

r = subprocess.run(
    [sys.executable, '-m', 'pyflakes', '--version'],
    capture_output=True, text=True, encoding='utf-8', errors='replace',
)
print('return code:', r.returncode)
print('STDOUT:', r.stdout)
if r.stderr:
    print('STDERR:', r.stderr)

return code: 0
STDOUT: 3.4.0 Python 3.11.9 on Windows



In [4]:
# Raw pyflakes on one file under redditwarp
import os
import subprocess
import sys

target = os.path.join(os.path.dirname(os.getcwd()), 'redditwarp', 'redditwarp', 'exceptions.py')
r = subprocess.run(
    [sys.executable, '-m', 'pyflakes', target],
    capture_output=True, text=True, encoding='utf-8', errors='replace',
)
print('return code:', r.returncode, '(1 = issues found, 0 = clean)')
print('--- raw stdout ---')
print(r.stdout, end='' if r.stdout else '(empty)\n')
if r.stderr:
    print('--- stderr ---')
    print(r.stderr)

return code: 0 (1 = issues found, 0 = clean)
--- raw stdout ---
(empty)


In [5]:
# Raw pyflakes on all package *.py files (batched for Windows argv limits)
import os
import subprocess
import sys
from pathlib import Path

rw = Path(os.path.dirname(os.getcwd())) / 'redditwarp' / 'redditwarp'
files = sorted(
    p for p in rw.rglob('*.py')
    if '__pycache__' not in p.parts
)
BATCH = 40
all_out: list[str] = []
codes: list[int] = []

for i in range(0, len(files), BATCH):
    batch = [str(p) for p in files[i : i + BATCH]]
    r = subprocess.run(
        [sys.executable, '-m', 'pyflakes', *batch],
        capture_output=True, text=True, encoding='utf-8', errors='replace',
    )
    codes.append(r.returncode)
    if r.stdout:
        all_out.append(r.stdout)
    if r.stderr:
        all_out.append(r.stderr)

combined = ''.join(all_out)
print('files checked:', len(files))
print('batch return codes:', codes, '(any 1 => at least one batch had findings)')
print('--- combined raw output (' + str(len(combined)) + ' chars) ---')
print(combined, end='' if combined else '(no pyflakes messages / all clean)\n')

files checked: 549
batch return codes: [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 1, 1] (any 1 => at least one batch had findings)
--- combined raw output (31533 chars) ---
F:\testable-whitebox-metrics\redditwarp\redditwarp\__init__.py:3:1: '.__about__' imported but unused
F:\testable-whitebox-metrics\redditwarp\redditwarp\__init__.py:4:1: '.__about__.__title__' imported but unused
F:\testable-whitebox-metrics\redditwarp\redditwarp\__init__.py:4:1: '.__about__.__summary__' imported but unused
F:\testable-whitebox-metrics\redditwarp\redditwarp\__init__.py:4:1: '.__about__.__uri__' imported but unused
F:\testable-whitebox-metrics\redditwarp\redditwarp\__init__.py:4:1: '.__about__.__version__' imported but unused
F:\testable-whitebox-metrics\redditwarp\redditwarp\__init__.py:4:1: '.__about__.__author__' imported but unused
F:\testable-whitebox-metrics\redditwarp\redditwarp\__init__.py:4:1: '.__about__.__license__' imported but unused
F:\testable-whitebox-metrics\redditwarp\redditwarp\__init__.p